# Retrain a Kapampangan plus Tagalog source tokenizer

Adds 993 Tagalog root entries and 7,880 Tagalog text records to the preserved 3,311 roots and 26,268 training records. CPU runtime is sufficient. This trains a tokenizer, not an NLLB translation model. Existing source embeddings cannot be reused with new token IDs.

## 1. Upload the prepared input ZIP
Upload `tagalog_30pct_tokenizer_inputs.zip`. All source code and prepared inputs are included. No dependency installation is needed.


In [ ]:
from pathlib import Path
import hashlib, json, subprocess, sys, zipfile
from google.colab import files

uploaded = files.upload()
archives = [Path(name) for name in uploaded if name.endswith('.zip')]
assert len(archives) == 1, 'Upload exactly the prepared tokenizer input ZIP.'
base = Path('/content/pam_tl_30pct_inputs')
assert not base.exists(), 'Use a fresh runtime or choose a new extraction folder.'
base.mkdir()
with zipfile.ZipFile(archives[0]) as archive:
    for member in archive.infolist():
        target = (base / member.filename).resolve()
        assert target.is_relative_to(base.resolve()), 'Unsafe archive member'
        assert not member.filename.startswith('/')
    archive.extractall(base)
packages = [p.parent for p in base.rglob('manifest.json') if (p.parent / 'train_tokenizer.py').is_file()]
assert len(packages) == 1
package = packages[0]
sys.path.insert(0, str(package))
from train_tokenizer import verify_inputs
manifest = verify_inputs(package)
print(json.dumps(manifest['counts'], indent=2))
print('All packaged input checksums passed.')


## 2. Choose the tokenizer and train
Default: weighted Morph-BPE, penalty32, vocabulary6,080.

For the separate infix-first48 experiment set `ALGORITHM='infix-first'` and `PENALTY=48`. For the matched Plain BPE control set `ALGORITHM='plain'`. Use a new `OUTPUT_NAME` for every run. Prepared inputs already apply each source language's own dictionary/rules.


In [ ]:
ALGORITHM = 'weighted'  # 'weighted', 'infix-first', or 'plain'
PENALTY = 32
VOCAB_SIZE = 6080
OUTPUT_NAME = f'pam_tl_30pct_{ALGORITHM}_p{PENALTY}_v{VOCAB_SIZE}'
output = Path('/content/tokenizer_runs') / OUTPUT_NAME
subprocess.run([
    sys.executable, str(package / 'train_tokenizer.py'),
    '--inputs', str(package), '--output', str(output),
    '--algorithm', ALGORITHM, '--penalty', str(PENALTY),
    '--vocab-size', str(VOCAB_SIZE),
], check=True)


## 3. Verify and download
The exported artifact contains the learned vocabulary and merge rules. Preserve it as a separate experimental tokenizer. Train new matching source embeddings before using it with NLLB. No translation-quality score is produced by this notebook.


In [ ]:
sys.path.insert(0, str(package / 'source'))
from kapampangan_morphbpe.artifact import validate_artifact_files
print(validate_artifact_files(output))
archive_path = output.parent / f'{OUTPUT_NAME}.zip'
with zipfile.ZipFile(archive_path, 'x', zipfile.ZIP_DEFLATED) as archive:
    for source in sorted(output.rglob('*')):
        if source.is_file():
            archive.write(source, source.relative_to(output))
print('SHA256:', hashlib.sha256(archive_path.read_bytes()).hexdigest())
files.download(str(archive_path))
